# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

Run the cells from top to bottom. The setup installs the classic chain APIs used in this lab. Set `OPENAI_API_KEY` in your environment or a local `.env` file, or enter it securely when prompted. Model calls use your API account. `OPENAI_MODEL` optionally overrides the default model.

The sequential review workflow translates the review, summarizes it, detects its original language, and uses the summary and language to write a reply.


In [ ]:
# Install into the active notebook kernel before importing dependencies.
%pip install "langchain-classic>=1,<2" "langchain-openai>=1,<2" "pandas>=2,<4" "python-dotenv>=1,<2"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

In [ ]:
# The dependency installation cell above includes pandas.

In [ ]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [ ]:
df.head()

## LLMChain

In [ ]:
# The lab uses legacy chain classes, now provided by langchain-classic.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [ ]:
# A moderate temperature gives varied marketing copy while keeping it focused.
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.7)

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Write a concise, appealing product description for {product}. "
    "Describe its purpose and likely benefits for the customer. "
    "Do not invent specifications, certifications, or guarantees."
)

In [ ]:

chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
product = "a reusable stainless steel water bottle"
chain.invoke({"product": product})["text"]

## SimpleSequentialChain

In [ ]:
from langchain_classic.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.9)

first_prompt = ChatPromptTemplate.from_template(
    "Suggest one memorable brand name for {product}. "
    "Return only the brand name."
)
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Write a short brand story for the brand named {brand_name}. "
    "Explain the values its name suggests and end with a catchy tagline. "
    "Treat this as fictional marketing copy."
)
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [ ]:
overall_simple_chain.invoke({"input": product})["output"]

**Repeat the above twice for different products**

In [ ]:
# Two additional products: each executes both chains.
simple_results = {}
for other_product in ["a solar-powered camping lantern", "a handmade ceramic coffee mug"]:
    result = overall_simple_chain.invoke({"input": other_product})
    simple_results[other_product] = result["output"]
    print(f"Product: {other_product}\n{result['output']}\n")


## SequentialChain

In [ ]:
from langchain_classic.chains import SequentialChain

In [ ]:
# Translation and summarization benefit from a low temperature.
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following customer review into English. "
    "If it is already English, return it unchanged. Return only the review."
    "\n\nReview: {Review}"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")

In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this English customer review in one sentence, preserving "
    "the sentiment and main praise or complaint.\n\n{English_Review}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")

In [ ]:
# Identify the original language so the reply can match the customer.
third_prompt = ChatPromptTemplate.from_template(
    "Identify the language of this customer review. "
    "Return only the language name.\n\n{Review}"
)
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

In [ ]:
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a polite customer-service follow-up in {language}, based on "
    "this review summary: {summary}. Acknowledge the customer's feedback "
    "and offer an appropriate next step. Do not promise refunds or "
    "replacements or invent company policies. Return only the message."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followup_message")

In [ ]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True,
)

In [ ]:
review = df.loc[5, "Review"]
overall_chain.invoke({"Review": review})

**Repeat the above twice for different products or reviews**

In [ ]:
# Two additional reviews, including Spanish and English examples.
review_results = []
for index in [6, 1]:
    result = overall_chain.invoke({"Review": df.loc[index, "Review"]})
    review_results.append(result)
    print(f"Product: {df.loc[index, 'Product']}")
    for key in overall_chain.output_keys:
        print(f"{key}: {result[key]}")
    print()


## Router Chain

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [ ]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [ ]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

In [ ]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": "one candidate name or DEFAULT",
    "next_inputs": "the original input or a clearer version"
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [ ]:
chain.invoke({"input": 'What is black body radiation?'})["text"]

In [ ]:
chain.invoke({"input": 'What is 2 + 2?'})["text"]

In [ ]:
chain.invoke({"input": 'Why do most cells in our body contain DNA, and which cells are exceptions?'})["text"]

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [ ]:
# Exercise the remaining experts and the default route.
router_questions = [
    "How did the printing press change the spread of ideas in Renaissance Europe?",
    "Explain binary search and its time complexity using a sorted bookshelf analogy.",
    "Write a four-line poem about a rainy afternoon.",
]
router_results = {}
for question in router_questions:
    result = chain.invoke({"input": question})
    router_results[question] = result["text"]
    print(f"Question: {question}\nAnswer: {result['text']}\n")
